train, valid, test로 평가해보기

In [55]:
import glob
from PIL import Image
import matplotlib.pyplot as plt

In [56]:
train_images = glob.glob("../Data/fingers/train/*.png")
test_images = glob.glob("../Data/fingers/test/*.png")

In [57]:
# train_data와 test data 만들기: 128*128 -> 32*32
train_input = []
train_target = []
test_input = []
test_target = []

# train data
for image in sorted(train_images):
    img = Image.open(image)
    imgResize = img.resize((32, 32), Image.Resampling.LANCZOS)
    train_input.append(imgResize)
    train_target.append(image[-6:-4])

# test data
for image in sorted(test_images):
    img = Image.open(image)
    imgResize = img.resize((32, 32), Image.Resampling.LANCZOS)
    test_input.append(imgResize)
    test_target.append(image[-6:-4])

In [58]:
# Target Data 확인
print(train_target[:5])
print(test_target[:5])

['0L', '0L', '2L', '0L', '5L']
['5L', '5L', '3R', '5L', '5L']


In [49]:
import numpy as np

In [59]:
# train Data 만들기 (18000 * 32 * 32)

tempData = np.zeros(
        18000 * 32 * 32,
        dtype=np.int32
).reshape(18000, 32, 32)

i = 0
for image in train_input:
    img = np.array(image, dtype=np.int32)
    tempData[i,:,:] = img
    i+=1

train_input = tempData.copy()

In [60]:
# test Data 만들기 (3600 * 32 * 32)

tempData = np.zeros(
        3600 * 32 * 32,
        dtype=np.int32
).reshape(3600, 32, 32)

i = 0
for image in test_input:
    img = np.array(image, dtype=np.int32)
    tempData[i,:,:] = img
    i+=1

test_input = tempData.copy()

In [61]:
# 배열 크기 확인
print(train_input.shape, test_input.shape)

(18000, 32, 32) (3600, 32, 32)


In [62]:
label_to_int = {
    '0R' : 0,
    '1R' : 1,
    '2R' : 2,
    '3R' : 3,
    '4R' : 4,
    '5R' : 5,
    '0L' : 6,
    '1L' : 7,
    '2L' : 8,
    '3L' : 9,
    '4L' : 10,
    '5L' : 11,
}

In [63]:
# Train_Target 숫자로 변경
temp = []
for label in train_target:
    temp.append(label_to_int[label])
train_target = temp.copy()

In [64]:
# Test_Target 숫자로 변경
temp = []
for label in test_target:
    temp.append(label_to_int[label])
test_target = temp.copy()

In [65]:
# Target도 numpy배열로 변경
train_target = np.array(train_target)
test_target = np.array(test_target)

train_target[:5]

array([ 6,  6,  8,  6, 11])

In [66]:
test_target

array([11, 11,  3, ...,  4,  3,  3], shape=(3600,))

In [67]:
train_scaled = train_input / 255.0

In [68]:
from tensorflow import keras
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models

 
train_scaled, val_scaled, train_target, val_target = \
    train_test_split(   
        train_scaled,
        train_target,
        test_size = 0.2,    
        random_state=42
    )

In [72]:
# 2. 흑백 이미지일 경우, 강제로 4차원 꼴 [개수, 32, 32, 1]로 변형합니다.
train_scaled = train_scaled.reshape(-1, 32, 32, 1)
val_scaled = val_scaled.reshape(-1, 32, 32, 1)

In [73]:
# 맨 뒤에 1차원(채널 축)을 강제로 추가합니다.
# (배치, 32, 32) -> (배치, 32, 32, 1) 형태로 변환됩니다.
train_scaled = np.expand_dims(train_scaled, axis=-1)
val_scaled = np.expand_dims(val_scaled, axis=-1)
#test_scaled = np.expand_dims(test_scaled, axis=-1)


# -------------------------------------------------------------
# 1. CNN 모델 구성 (합성곱 신경망)
# -------------------------------------------------------------
model_cnn = models.Sequential()

# [Feature Extraction 단계] 이미지의 공간적 특징 추출
# Conv2D: 32개의 3x3 필터를 사용해 특징을 찾고, ReLU 활성화 함수 적용
model_cnn.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32,  32, 1)))
model_cnn.add(layers.MaxPooling2D((2, 2))) # 이미지 크기를 줄여 핵심만 남김 (다운샘플링)

model_cnn.add(layers.Conv2D(64, (3, 3), activation='relu'))
model_cnn.add(layers.MaxPooling2D((2, 2)))

model_cnn.add(layers.Conv2D(64, (3, 3), activation='relu'))

# [Classification 단계] 추출된 특징을 바탕으로 다중 분류 진행
model_cnn.add(layers.Flatten()) # 추출된 2차원 특징 맵들을 1차원으로 쫙 펼치기
model_cnn.add(layers.Dense(64, activation='relu'))
model_cnn.add(layers.Dropout(0.2)) # 과적합 방지

# 최종 출력층 (클래스 개수 12개, Softmax)
model_cnn.add(layers.Dense(12, activation='softmax'))

In [74]:
# -------------------------------------------------------------
# 2. 모델 컴파일
# -------------------------------------------------------------
model_cnn.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model_cnn.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_15 (Conv2D)              │ (None, 30, 30, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 15, 15, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_16 (Conv2D)              │ (None, 13, 13, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_11 (MaxPooling2D) │ (None, 6, 6, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_17 (Conv2D)              │ (None, 4, 4, 64)       │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_5 (Flatten)             │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 64)             │        65,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 12)             │           780 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 122,124 (477.05 KB)

 Trainable params: 122,124 (477.05 KB)

 Non-trainable params: 0 (0.00 B)

In [75]:
# -------------------------------------------------------------
# 3. 모델 학습 (validation 데이터 활용)
# -------------------------------------------------------------
history = model_cnn.fit(
    train_scaled, 
    train_target, 
    epochs=15,                
    batch_size=32,
    validation_data=(val_scaled, val_target)
)

Epoch 1/15
450/450 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.8584 - loss: 0.4427 - val_accuracy: 0.9911 - val_loss: 0.0230
Epoch 2/15
450/450 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9912 - loss: 0.0264 - val_accuracy: 1.0000 - val_loss: 0.0016
Epoch 3/15
450/450 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.9970 - loss: 0.0110 - val_accuracy: 0.9989 - val_loss: 0.0025
Epoch 4/15
450/450 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9990 - loss: 0.0049 - val_accuracy: 1.0000 - val_loss: 2.4629e-04
Epoch 5/15
450/450 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9982 - loss: 0.0051 - val_accuracy: 1.0000 - val_loss: 9.9057e-05
Epoch 6/15
450/450 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9977 - loss: 0.0070 - val_accuracy: 1.0000 - val_loss: 2.9734e-05
Epoch 7/15
450/450 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9988 - loss: 0.0035 - val_accuracy: 1.0000 - val_loss: 1.4342e-04
Epoch 8/15
450/450 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.9994 - loss: 0.0021 - 

In [76]:
test_scaled = test_input / 255.0

test_scaled = test_scaled.reshape(-1, 32, 32, 1)
test_scaled = np.expand_dims(test_scaled, axis=-1)

# 역방향 딕셔너리 준비
int_to_label = {v: k for k, v in label_to_int.items()}

# 1. CNN 모델로 테스트 데이터셋 전체 예측 수행
predictions_cnn = model_cnn.predict(test_scaled)

# 2. 가장 높은 확률을 가진 인덱스 추출
predicted_labels_cnn = np.argmax(predictions_cnn, axis=1)

# 3. 숫자를 문자열 레이블로 변환
predicted_string_labels = [int_to_label[idx] for idx in predicted_labels_cnn]
real_string_labels = [int_to_label[idx] for idx in test_target]

# 4. 결과 출력 (상위 10개 결과 대조)
print("\n" + "="*50)
print("     [실제 정답]  vs  [CNN 모델의 예측값]")
print("="*50)
for i in range(10):
    print(f"샘플 [{i}]: 실제 정답 = {real_string_labels[i]:<4} | CNN 예측 = {predicted_string_labels[i]}")
print("="*50)

113/113 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step

     [실제 정답]  vs  [CNN 모델의 예측값]
샘플 [0]: 실제 정답 = 5L   | CNN 예측 = 5L
샘플 [1]: 실제 정답 = 5L   | CNN 예측 = 5L
샘플 [2]: 실제 정답 = 3R   | CNN 예측 = 3R
샘플 [3]: 실제 정답 = 5L   | CNN 예측 = 5L
샘플 [4]: 실제 정답 = 5L   | CNN 예측 = 5L
샘플 [5]: 실제 정답 = 4R   | CNN 예측 = 4R
샘플 [6]: 실제 정답 = 2R   | CNN 예측 = 2R
샘플 [7]: 실제 정답 = 4R   | CNN 예측 = 4R
샘플 [8]: 실제 정답 = 5L   | CNN 예측 = 5L
샘플 [9]: 실제 정답 = 4R   | CNN 예측 = 4R


In [77]:
model_cnn.evaluate(test_scaled, test_target)

113/113 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 1.0000 - loss: 3.6351e-06


[3.635092298281961e-06, 1.0]